# EDA - Weather

In [8]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent

INPUT_PATH = BASE_DIR / "data/raw/weather/weather_forecasts.csv"
OUTPUT_PATH = BASE_DIR / "data/outputs/weather_cleaned.csv"

In [9]:
df = pd.read_csv(INPUT_PATH)
df.head()

,city,datetime,temperature,feels_like,humidity,weather,wind_speed,rain_3h
0,Mont Saint Michel,2026-04-07 15:00:00,22.27,22.43,72,overcast clouds,5.95,0.0
1,Mont Saint Michel,2026-04-07 18:00:00,21.08,21.20,75,overcast clouds,3.51,0.0
2,Mont Saint Michel,2026-04-07 21:00:00,17.37,17.25,80,broken clouds,3.42,0.0
3,Mont Saint Michel,2026-04-08 00:00:00,13.79,13.47,86,overcast clouds,2.74,0.0
4,Mont Saint Michel,2026-04-08 03:00:00,12.66,12.33,90,overcast clouds,2.64,0.0


## Duplicates

In [10]:
df_dup = df[df.duplicated()]
print(df_dup)

Empty DataFrame
Columns: [city, datetime, temperature, feels_like, humidity, weather, wind_speed, rain_3h]
Index: []


The output shows that the dataset currently contains no duplicate rows.

However, to ensure data quality in future extractions, a duplicate removal step is included in the transformation pipeline.  
This operation removes fully duplicated rows from the dataset before loading the cleaned data into the data warehouse.

It could be as follows : 

In [11]:
df = df.drop_duplicates()

## Datetime conversion

In [14]:
df["datetime"].dtype

dtype('<M8[us]')

The `datetime` column contains the date and time of each weather forecast.

Converting this column to a proper datetime format is important because it allows pandas to correctly interpret temporal information and enables   time-based operations.  

In [16]:
df["datetime"] = pd.to_datetime(df["datetime"])
print(df["datetime"].dtype)

datetime64[us]


After conversion, the column datatype becomes:
datetime64[us]

This indicates that pandas now stores the values as true datetime objects with microsecond precision instead of plain text values.

## Numeric conversions

In [19]:
numeric_cols = [
    "temperature",
    "feels_like",
    "humidity",
    "wind_speed",
    "rain_3h"
]

for col in numeric_cols : 
    print(col, ":", df[col].dtype)


temperature : float64
feels_like : float64
humidity : int64
wind_speed : float64
rain_3h : float64


In this extraction, the columns were already interpreted by pandas as numerical datatypes (`float64` and `int64`).

However, explicit numeric conversion is still included in the transformation pipeline to ensure robustness and consistency  
for future raw datasets that may contain malformed or non-numeric values.

In [20]:
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in numeric_cols : 
    print(col, ":", df[col].dtype)

temperature : float64
feels_like : float64
humidity : int64
wind_speed : float64
rain_3h : float64


## Null values

In [21]:
(df.isna().mean() * 100).sort_values(ascending=False)

city           0.0
datetime       0.0
temperature    0.0
feels_like     0.0
humidity       0.0
weather        0.0
wind_speed     0.0
rain_3h        0.0
dtype: float64

The results show that the current weather dataset contains no missing values.
Therefore, no imputation or row deletion was also required for this extraction.
However, defensive cleaning rules are still considered for future raw datasets in order to make the ETL pipeline more robust.

Different missing value strategies are applied depending on the business meaning of each column.

- Rows missing `city` or `datetime` are removed because these fields are essential identifiers for the weather forecasts.
- Missing values in `rain_3h` are replaced with `0`, since the absence of rain information from the API is interpreted as no expected rainfall.
- Numerical weather variables (`temperature`, `feels_like`, `humidity`, `wind_speed`) are filled using the median value of their respective columns in order to preserve the dataset while limiting the impact of extreme values.

In [22]:
# Removing rows with critical missing identifiers
df = df.dropna(subset=["city", "datetime"])

# Filling rain values with 0 because missing rain information is interpreted as no expected rainfall
df["rain_3h"] = df["rain_3h"].fillna(0)

# Filling the other numerical weather columns with median values
numeric_fill_cols = [
    "temperature",
    "feels_like",
    "humidity",
    "wind_speed"
]

for col in numeric_fill_cols:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)


print((df.isna().mean() * 100).sort_values(ascending=False))

city           0.0
datetime       0.0
temperature    0.0
feels_like     0.0
humidity       0.0
weather        0.0
wind_speed     0.0
rain_3h        0.0
dtype: float64
